# 01 · Clean and merge

Merges the Kaggle *U.S. Banks Financial Quarterly Ratios* file (`data/total_bank_data.csv`) with the FRED Federal Funds Rate (`data/FEDFUNDS.csv`) on quarter, builds the control variables, winsorizes outliers at 1.5×IQR fences, and exports `data/final_dataset.csv` (639,871 bank-quarter rows).

> Note: the IQR step is applied twice (once in section 10 and again in section 11). The second pass recomputes the quartiles on already-clipped data and tightens the fences very slightly. All results in notebooks 02 and 03 were produced from the file this exact sequence generates.


In [1]:
# === 1. Core Libraries ===
import pandas as pd
import numpy as np

# === 2. Visualization ===
import matplotlib.pyplot as plt
import seaborn as sns

# === 3. Statistical Tools ===
from scipy import stats

# === 4. Display Settings ===
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)
sns.set(context="notebook", style="whitegrid")

In [2]:
# === 5. File Paths ===
KAGGLE_PATH = "data/total_bank_data.csv"
FRED_PATH   = "data/FEDFUNDS.csv"

# === 6. Load Raw Data ===
banks = pd.read_csv(KAGGLE_PATH)
fred  = pd.read_csv(FRED_PATH)

# Quick structure check
banks.head(), fred.head(), (banks.shape, fred.shape)

(   ABCUBKR  ABCUOTHR  ABCXBKR  ABCXOTHR  ADDNONINTEXPQR  ADDNONINTEXPR  ALLOTHLR  ASCEOTHR  ASCERESR  ASDROTHR  \
 0      0.0       0.0      0.0       0.0        0.896543       0.851559  0.363534       0.0       0.0       0.0   
 1      0.0       0.0      0.0       0.0        0.688036       0.650281  0.310936       0.0       0.0       0.0   
 2      0.0       0.0      0.0       0.0        0.647623       0.619826  0.149273       0.0       0.0       0.0   
 3      0.0       0.0      0.0       0.0        1.064997       1.034980  0.222062       0.0       0.0       0.0   
 4      0.0       0.0      0.0       0.0        0.631587       0.595708  0.735652       0.0       0.0       0.0   
 
    ASDRRESR    ASSTLTR   AVASSETJR   BKPREMR   BROINSR      BROR  CBLRIND  CBLRINDQ    CD1T3R   CD1T3SR   CD3LESR  \
 0       0.0  31.232309   99.616119  0.585996  8.472975  8.472975      0.0       NaN  0.238739  6.801801  0.000000   
 1       0.0  28.480927   98.554465  1.682011  0.983983  0.983983      0

In [3]:
# === 7. Build quarterly keys and FEDFUNDS_CHANGE ===

# Banks: REPDTE -> datetime -> quarter
banks['REPDTE_dt'] = pd.to_datetime(banks['REPDTE'], format='%Y%m%d', errors='coerce')
banks['QUARTER']   = banks['REPDTE_dt'].dt.to_period('Q')

# FRED: observation_date -> datetime -> quarter
fred['observation_date_dt'] = pd.to_datetime(fred['observation_date'], errors='coerce')
fred['QUARTER']             = fred['observation_date_dt'].dt.to_period('Q')
fred['FEDFUNDS']            = pd.to_numeric(fred['FEDFUNDS'], errors='coerce')

# Keep only what we need and compute Δ
fred_q = (
    fred[['QUARTER', 'FEDFUNDS']].dropna()
         .sort_values('QUARTER')
         .assign(FEDFUNDS_CHANGE=lambda d: d['FEDFUNDS'].diff())
)

# === 8. Merge to bank panel ===
panel = banks.merge(fred_q, on='QUARTER', how='inner')

# Quick sanity checks
print("Banks quarters:", banks['QUARTER'].min(), "→", banks['QUARTER'].max(), "n=", banks['QUARTER'].nunique())
print("FRED  quarters:", fred_q['QUARTER'].min(),  "→", fred_q['QUARTER'].max(),  "n=", fred_q['QUARTER'].nunique())
panel[['QUARTER','FEDFUNDS','FEDFUNDS_CHANGE']].drop_duplicates().head()

Banks quarters: 2001Q1 → 2023Q2 n= 90
FRED  quarters: 2001Q1 → 2023Q1 n= 89


,QUARTER,FEDFUNDS,FEDFUNDS_CHANGE
0,2023Q1,4.52,0.87
4740,2022Q4,3.65,1.46
9513,2022Q3,2.19,1.42
14326,2022Q2,0.77,0.65
19164,2022Q1,0.12,0.04


In [4]:
# === 9. Create Control Variables Based on Ratio Metrics ===

panel['equity_ratio']  = pd.to_numeric(panel['EQTOTR'],  errors='coerce')
panel['loan_ratio']    = pd.to_numeric(panel['LNLSNETR'], errors='coerce')
panel['deposit_ratio'] = pd.to_numeric(panel['DEPR'],     errors='coerce')
panel['funding_ratio'] = pd.to_numeric(panel['LIABR'],    errors='coerce')

# --- composite: deposit share of funding ---
panel['deposit_funding_ratio'] = panel['deposit_ratio'] / panel['funding_ratio']

# --- Quick check ---
panel[['equity_ratio', 'loan_ratio', 'deposit_ratio', 
       'funding_ratio', 'deposit_funding_ratio']].describe().T

,count,mean,std,min,25%,50%,75%,max
equity_ratio,651724.0,12.373183,10.488558,-519.469027,8.720928,10.255883,12.548608,100.049505
loan_ratio,651724.0,61.636943,18.019472,-2.099958,52.347361,64.715086,74.410335,99.942409
deposit_ratio,651724.0,81.267653,13.565357,0.000000,79.264969,84.556201,88.168684,115.190525
funding_ratio,651724.0,87.626808,10.488561,0.000000,87.451380,89.744137,91.279057,619.469027
deposit_funding_ratio,651576.0,0.918455,0.137313,0.000000,0.899030,0.959321,0.990907,1.004692


In [5]:
# === 10. Add Profitability Variables ===

panel['ROAQ']  = pd.to_numeric(panel['ROAQ'],  errors='coerce')
panel['ROEQ']  = pd.to_numeric(panel['ROEQ'],  errors='coerce')
panel['NIMQR'] = pd.to_numeric(panel['NIMQR'], errors='coerce')

# --- Quick check ---
panel[['ROAQ', 'ROEQ', 'NIMQR']].describe().T

,count,mean,std,min,25%,50%,75%,max
ROAQ,650086.0,0.009304,0.054330,-7.619000,0.004900,0.009200,0.013600,6.794900
ROEQ,649933.0,0.070375,0.706238,-360.307200,0.043400,0.086100,0.131400,91.980300
NIMQR,651724.0,3.525650,1.173357,-160.881543,3.025102,3.496633,3.983018,70.680024


In [6]:
# Combine profitability + controls
cols_to_clean = ['ROAQ','ROEQ','NIMQR',
                 'equity_ratio','loan_ratio','deposit_ratio','funding_ratio','deposit_funding_ratio']

def iqr_trim(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return series.clip(q1 - k*iqr, q3 + k*iqr)

panel[cols_to_clean] = panel[cols_to_clean].apply(iqr_trim)
panel = panel.dropna(subset=cols_to_clean + ['FEDFUNDS','FEDFUNDS_CHANGE','CERT','QUARTER']).drop_duplicates()

panel[cols_to_clean].describe().T

,count,mean,std,min,25%,50%,75%,max
ROAQ,639871.0,0.009073,0.007850,-0.008150,0.004900,0.009200,0.013600,0.026650
ROEQ,639871.0,0.086202,0.077923,-0.088600,0.043200,0.085900,0.131200,0.263400
NIMQR,639871.0,3.506604,0.784217,1.588230,3.027451,3.495768,3.980085,5.419890
equity_ratio,639871.0,10.988246,3.214068,2.979408,8.744333,10.271565,12.557406,18.290127
loan_ratio,639871.0,62.110188,16.700312,19.252899,52.386016,64.747872,74.437368,99.929453
deposit_ratio,639871.0,82.878117,7.055279,65.909396,79.316202,84.570609,88.169268,101.524256
funding_ratio,639871.0,89.011747,3.214064,81.709866,87.442500,89.728459,91.255615,97.020572
deposit_funding_ratio,639871.0,0.934200,0.069404,0.761213,0.899634,0.959485,0.990977,1.004692


In [7]:
# === 11. Combine Profitability + Controls and Clean Data ===

cols_to_clean = [
    'ROAQ', 'ROEQ', 'NIMQR',
    'equity_ratio', 'loan_ratio',
    'deposit_ratio', 'funding_ratio', 'deposit_funding_ratio'
]

# --- IQR-based outlier trimming ---
def iqr_trim(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return series.clip(q1 - k * iqr, q3 + k * iqr)

panel[cols_to_clean] = panel[cols_to_clean].apply(iqr_trim)

# --- Drop missing and duplicate observations ---
panel = panel.dropna(subset=cols_to_clean + ['FEDFUNDS', 'FEDFUNDS_CHANGE', 'CERT', 'QUARTER'])
panel = panel.drop_duplicates()

# --- Check cleaned data ---
panel[cols_to_clean].describe().T

,count,mean,std,min,25%,50%,75%,max
ROAQ,639871.0,0.009073,0.007850,-0.008150,0.004900,0.009200,0.013600,0.026650
ROEQ,639871.0,0.086196,0.077910,-0.088600,0.043200,0.085900,0.131200,0.263200
NIMQR,639871.0,3.506598,0.783101,1.598502,3.027451,3.495768,3.980085,5.409035
equity_ratio,639871.0,10.987496,3.211663,3.024723,8.744333,10.271565,12.557406,18.277016
loan_ratio,639871.0,62.111889,16.695947,19.308987,52.386016,64.747872,74.437368,99.929453
deposit_ratio,639871.0,82.885387,7.037828,66.036603,79.316202,84.570609,88.169268,101.448867
funding_ratio,639871.0,89.012487,3.211683,81.722827,87.442500,89.728459,91.255615,96.975287
deposit_funding_ratio,639871.0,0.934279,0.069206,0.762619,0.899634,0.959485,0.990977,1.004692


In [8]:
# === 12. Trim Dataset to Essential Variables and Export Clean File ===

# --- Keep only relevant variables for analysis ---
keep_vars = [
    'CERT', 'QUARTER',                      # identifiers
    'ROAQ', 'ROEQ', 'NIMQR',                # dependent variables
    'FEDFUNDS', 'FEDFUNDS_CHANGE',          # main independent variables
    'equity_ratio', 'loan_ratio', 'deposit_ratio',
    'funding_ratio', 'deposit_funding_ratio'  # control variables
]

panel = panel[keep_vars].copy()

# --- Light cleanup before export ---
panel['QUARTER'] = panel['QUARTER'].astype(str)  # convert Period → string for CSV
panel['CERT'] = pd.to_numeric(panel['CERT'], errors='coerce').astype('Int64')

# --- Save final clean dataset ---
output_path = "data/final_dataset.csv"
panel.to_csv(output_path, index=False)

print("✅ Saved clean dataset:", output_path)
print("Shape:", panel.shape)
print("Columns:", list(panel.columns))
print("Approx. size (MB):", round(panel.memory_usage(deep=True).sum() / (1024**2), 2))

✅ Saved clean dataset: data/final_dataset.csv
Shape: (639871, 12)
Columns: ['CERT', 'QUARTER', 'ROAQ', 'ROEQ', 'NIMQR', 'FEDFUNDS', 'FEDFUNDS_CHANGE', 'equity_ratio', 'loan_ratio', 'deposit_ratio', 'funding_ratio', 'deposit_funding_ratio']
Approx. size (MB): 92.75
